# EAIOS M7 · real-model smoke test (verified report → approved draft → Mailpit)

Choose **Runtime → Change runtime type → T4 GPU**, then run the cells in order. The model gateway, backend, Celery worker and Mailpit all run in this runtime over loopback; no tunnel is opened and no email leaves the runtime. The final cell downloads the secret-scanned evidence bundle `m7-smoke-evidence-<stamp>.zip`.


In [ ]:
# @title 1 · Checkout the branch under review and verify the GPU
import os, subprocess
REPO = "/content/Enterprise-AI-OS"
BRANCH = "feature/m7-functional-acceptance"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/youssefhatembahig6-alt/Enterprise-AI-OS", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", "FETCH_HEAD"], check=True)
subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%H %s"], check=True)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)


In [ ]:
# @title 2 · Install the pinned vLLM environment
subprocess.run(["python", f"{REPO}/ml/colab-m7-acceptance/run_m7_smoke_colab.py", "vllm-env"], check=True)


In [ ]:
# @title 3 · Launch the model gateway on loopback
import secrets
os.environ.setdefault("GATEWAY_TOKEN", secrets.token_urlsafe(32))
%run /content/Enterprise-AI-OS/ml/model-gateway/colab_launch.py


In [ ]:
# @title 4 · Run the smoke test and bundle evidence
subprocess.run(["python", f"{REPO}/ml/colab-m7-acceptance/run_m7_smoke_colab.py", "all"], check=True)


In [ ]:
# @title 5 · Download the secret-scanned evidence bundle
import glob
from google.colab import files
bundles = sorted(glob.glob("/content/m7-smoke-evidence-*.zip"))
if not bundles:
    raise RuntimeError("No evidence bundle exists; inspect the output of cell 4.")
files.download(bundles[-1])
